# Make Chemical Checker Sign3 parquet

In [ ]:
# @title
import os
import glob
import re
import h5py
import pandas as pd
import numpy as np
import gc
import subprocess

# =====================================================================
# 1. GCS CONFIGURATION & WORKSPACE SETUP
# =====================================================================
gcs_input_dir = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign3_signatures'
gcs_output_dir = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign3_parquets'

local_h5_dir = '/content/local_sign3_h5'
os.makedirs(local_h5_dir, exist_ok=True)

print("🔍 Scanning GCS bucket for .h5 files...")
ls_output = subprocess.check_output(f"gsutil ls {gcs_input_dir}/*.h5", shell=True, text=True)
all_gcs_files =[f.strip() for f in ls_output.strip().split('\n') if f.strip()]

print(f"✅ Found {len(all_gcs_files)} .h5 files to process.")

# =====================================================================
# 2. PROCESS ONE FILE AT A TIME (Zero RAM Spikes)
# =====================================================================
for gcs_file in all_gcs_files:
    filename = os.path.basename(gcs_file)

    # Safely extract space name (e.g. 'A1')
    match = re.search(r'([A-E][1-5])', filename.upper())
    if not match:
        continue
    space_name = match.group(1)

    gcs_parquet = f"{gcs_output_dir}/{space_name}_sign3.parquet"
    local_parquet = f"/content/{space_name}_sign3.parquet"
    local_h5 = f"{local_h5_dir}/{filename}"

    # SAFETY CHECK: Skip if already converted
    check_exists = subprocess.run(f"gsutil -q stat {gcs_parquet}", shell=True)
    if check_exists.returncode == 0:
        print(f"⏭️ {space_name} already exists in Cloud Storage! Skipping...")
        continue

    print(f"\n🚀 PROCESSING SPACE {space_name}...")

    # 1. Download just THIS ONE file
    print(f"   📥 Downloading {filename}...")
    os.system(f"gsutil cp {gcs_file} {local_h5}")

    # 2. Extract Data (Using numpy string conversion is much faster/lighter than list comprehensions)
    print(f"   🧠 Loading into RAM...")
    with h5py.File(local_h5, 'r') as f:
        data_v = f['V'][()].astype(np.float32)
        # .astype(str) handles byte decoding safely and efficiently
        keys = np.array(f['keys']).astype(str)

    num_cols = data_v.shape[1]
    col_names = [f"{space_name}_Dim_{i+1}" for i in range(num_cols)]

    # 3. Build DataFrame and Save
    print(f"   💾 Saving to local Parquet...")
    df = pd.DataFrame(data_v, columns=col_names)
    df.insert(0, 'InChIKey', keys)
    df.to_parquet(local_parquet, compression='snappy', index=False)

    # 4. Upload to GCS
    print(f"   📤 Uploading {space_name} to Google Cloud Storage...")
    os.system(f"gsutil cp {local_parquet} {gcs_parquet}")

    # 5. Aggressive Cleanup
    print("   🧹 Cleaning up local SSD and RAM...")
    os.remove(local_h5)
    os.remove(local_parquet)
    del data_v, keys, df
    gc.collect()

print("\n🎉 ALL SPACES SUCCESSFULLY CONVERTED TO PARQUET!")

In [ ]:
# @title
!pip install duckdb -q
import os
import duckdb
import time
import glob
import shutil
import pandas as pd

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
metadata_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'
gcs_parquet_dir = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign3_parquets'
gcs_final_output = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/parquets/JUMPCP_Sign3_CC_final.parquet'

local_parquet_dir = '/content/local_sign3_parquets'
master_accum_file = '/content/master_accumulator.parquet'
temp_accum_file = '/content/temp_accumulator.parquet'
duckdb_temp_dir = '/content/duckdb_temp'

os.makedirs(local_parquet_dir, exist_ok=True)
os.makedirs(duckdb_temp_dir, exist_ok=True)

# =====================================================================
# 2. EXTRACT METADATA SKELETON (WITH ALL COLUMNS)
# =====================================================================
print("📂 Loading full metadata from CSV...")
df_csv = pd.read_csv(metadata_csv)

# Auto-detect the InChIKey column
inchikey_col = next((col for col in df_csv.columns if 'inchikey' in col.lower()), None)
if not inchikey_col:
    raise KeyError("❌ Could not find an 'InChIKey' column in your CSV. Please check your headers!")

# 1. Rename it to exactly 'InChIKey' so the DuckDB join logic perfectly aligns
df_csv.rename(columns={inchikey_col: 'InChIKey'}, inplace=True)

# 2. Drop rows missing the key, strip whitespace, and deduplicate
# (Deduplication prevents exponential row explosion during joins!)
df_csv = df_csv.dropna(subset=['InChIKey'])
df_csv['InChIKey'] = df_csv['InChIKey'].astype(str).str.strip()
df_csv = df_csv.drop_duplicates(subset=['InChIKey'], keep='first')

# 3. Clean mixed data types to prevent PyArrow compression crashes
object_columns = df_csv.select_dtypes(include=['object']).columns
for col in object_columns:
    df_csv[col] = df_csv[col].astype(str)

print(f"✅ Extracted {len(df_csv):,} unique target molecules with ALL metadata columns.")

# Save this rich metadata table as our starting "Master File"
df_csv.to_parquet(master_accum_file, compression='snappy', index=False)
del df_csv # Free up RAM!

# =====================================================================
# 3. STAGE PARQUETS LOCALLY
# =====================================================================
print("\n📥 Pulling 25 Parquets from GCS to local SSD... (Fast multithreaded download)")
os.system(f"gsutil -m cp {gcs_parquet_dir}/*_sign3.parquet {local_parquet_dir}/")

parquet_spaces = sorted(glob.glob(f"{local_parquet_dir}/*_sign3.parquet"))
print(f"   ✅ Downloaded {len(parquet_spaces)} files for assembly.")

if len(parquet_spaces) == 0:
    raise FileNotFoundError("No parquet files found! Check your GCS path.")

# =====================================================================
# 4. THE STRICT 2-WAY ACCUMULATOR LOOP (Zero RAM Crashes)
# =====================================================================
print("\n🚀 Starting Incremental 2-Way INNER Joins...")
overall_start_time = time.time()

con = duckdb.connect()
con.execute("PRAGMA memory_limit='4GB'") # Force early SSD spilling to avoid pinning errors
con.execute(f"PRAGMA temp_directory='{duckdb_temp_dir}'")
con.execute("PRAGMA preserve_insertion_order=false")

for i, f in enumerate(parquet_spaces):
    space_name = os.path.basename(f).replace('_sign3.parquet', '')
    print(f"   [{i+1}/{len(parquet_spaces)}] Merging Space {space_name}...")

    step_start = time.time()

    # 🚨 m.* carries ALL of your metadata columns forward through every single loop!
    query = f"""
    COPY (
        SELECT
            m.*,
            r.* EXCLUDE (InChIKey)
        FROM read_parquet('{master_accum_file}') m
        INNER JOIN read_parquet('{f}') r
        ON m.InChIKey = r.InChIKey
    ) TO '{temp_accum_file}' (FORMAT PARQUET, COMPRESSION 'snappy');
    """

    try:
        con.execute(query)
    except Exception as e:
        con.close()
        raise RuntimeError(f"DuckDB Merge Error on {space_name}: {e}")

    # Atomically replace the old master with the newly merged file
    os.replace(temp_accum_file, master_accum_file)

    current_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{master_accum_file}')").fetchone()[0]
    print(f"      -> Done in {time.time() - step_start:.1f}s | {current_rows:,} molecules remain.")

con.close()
print(f"\n✅ All 25 spaces successfully merged in {time.time() - overall_start_time:.1f} seconds!")

# =====================================================================
# 5. VERIFY, UPLOAD & CLEANUP
# =====================================================================
print("\n📊 Verifying the dense assembled dataset...")
df_assembled = pd.read_parquet(master_accum_file)

print("-" * 65)
print(f"   Molecules (Rows):     {df_assembled.shape[0]:,} (Dense Data Only!)")
print(f"   Dimensions (Columns): {df_assembled.shape[1]:,}")
print("-" * 65)

# Show a quick preview to confirm metadata + embeddings are there
display(df_assembled.head(3))

print("\n📤 Uploading fully assembled annotated dataset to Google Cloud Storage...")
os.system(f"gsutil cp {master_accum_file} {gcs_final_output}")

print(f"🎉 SUCCESS! Annotated Global Signatures saved permanently to:")
print(f"   {gcs_final_output}")

print("\n🧹 Cleaning up local SSD workspace...")
shutil.rmtree(local_parquet_dir, ignore_errors=True)
shutil.rmtree(duckdb_temp_dir, ignore_errors=True)
if os.path.exists(master_accum_file): os.remove(master_accum_file)

print("Done! You are ready to analyze the data.")

# Make Chemical Checker Sign2 parquet

In [ ]:
# @title
!pip install duckdb gcsfs -q
import os
import duckdb
import pandas as pd
import time
import shutil

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
csv_path = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

# Output location requested
gcs_output_parquet = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMPCP_CC_Sign2_Global_Signatures.parquet'

local_parquet_dir = '/content/local_sign2_parquets'
local_output_parquet = '/content/JUMPCP_CC_Sign2_Global_Signatures.parquet'
duckdb_temp_dir = '/content/duckdb_temp'

os.makedirs(local_parquet_dir, exist_ok=True)
os.makedirs(duckdb_temp_dir, exist_ok=True)

# Clean up any leftover files from previous runs
if os.path.exists(local_output_parquet): os.remove(local_output_parquet)

# =====================================================================
# 2. LOAD TARGET INCHIKEYS FROM CSV
# =====================================================================
print(f"📂 Loading target subset from {csv_path}...")
df_csv = pd.read_csv(csv_path)

# Auto-detect the InChIKey column (e.g., 'Metadata_InChIKey', 'InChIKey')
inchikey_col = next((col for col in df_csv.columns if 'inchikey' in col.lower()), None)
if not inchikey_col:
    raise KeyError("❌ Could not find an 'InChIKey' column in your CSV. Please check your headers!")

# Extract unique target keys
target_inchikeys = df_csv[inchikey_col].dropna().astype(str).str.strip().unique()
print(f"✅ Extracted {len(target_inchikeys):,} unique target InChIKeys.")

# Wrap in a lightweight DataFrame for DuckDB to use as the Driving Table
df_targets = pd.DataFrame({'Target_InChIKey': target_inchikeys})

# =====================================================================
# 3. STAGE PARQUETS LOCALLY (Bypasses GCS Network Throttling)
# =====================================================================
print("\n📥 Pulling Sign2 Parquets from GCS to local SSD... (Fast multithreaded download)")
# We pull all 5 blocks at once to the local disk to make the SQL join lightning fast
os.system(f"gsutil -m cp gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/sign2_signatures/Block_*.parquet {local_parquet_dir}/")

# Map the local paths
blk_A = f"{local_parquet_dir}/Block_A_signatures.parquet"
blk_B = f"{local_parquet_dir}/Block_B_signatures.parquet"
blk_C = f"{local_parquet_dir}/Block_C_signatures.parquet"
blk_D = f"{local_parquet_dir}/Block_D_signatures.parquet"
blk_E = f"{local_parquet_dir}/Block_E_signatures.parquet"

# =====================================================================
# 4. EXECUTE THE INTERSECTION & SEQUENTIAL LEFT JOINS
# =====================================================================
print("\n⚙️ Executing the out-of-core streaming join...")
start_time = time.time()

con = duckdb.connect()
con.execute("PRAGMA memory_limit='8GB'")
con.execute(f"PRAGMA temp_directory='{duckdb_temp_dir}'")

# The SQL Logic:
# 1. INNER JOIN between your CSV targets and Block A creates the rigid intersection.
# 2. SEQUENTIAL LEFT JOINS pull in B, C, D, and E. If a molecule exists in A but
#    is missing in C, the columns for C are safely populated with NULLs (NaNs).
# 3. EXCLUDE (InChIKey) prevents Pandas from creating duplicate columns (InChIKey_1, etc.)

query = f"""
COPY (
    SELECT
        a.*,
        b.* EXCLUDE (InChIKey),
        c.* EXCLUDE (InChIKey),
        d.* EXCLUDE (InChIKey),
        e.* EXCLUDE (InChIKey)
    FROM df_targets t
    INNER JOIN read_parquet('{blk_A}') a ON t.Target_InChIKey = a.InChIKey
    LEFT JOIN read_parquet('{blk_B}') b ON a.InChIKey = b.InChIKey
    LEFT JOIN read_parquet('{blk_C}') c ON a.InChIKey = c.InChIKey
    LEFT JOIN read_parquet('{blk_D}') d ON a.InChIKey = d.InChIKey
    LEFT JOIN read_parquet('{blk_E}') e ON a.InChIKey = e.InChIKey
) TO '{local_output_parquet}' (FORMAT PARQUET, COMPRESSION 'snappy');
"""

try:
    con.execute(query)
    print(f"  ✅ Join completed flawlessly in {time.time() - start_time:.1f} seconds!")
except Exception as e:
    raise RuntimeError(f"DuckDB Merge Error: {e}")
finally:
    con.close()

# =====================================================================
# 5. UPLOAD TO GCS & VERIFY
# =====================================================================
print("\n📤 Uploading final assembled dataset to Google Cloud Storage...")
os.system(f"gsutil cp {local_output_parquet} {gcs_output_parquet}")

print(f"🎉 SUCCESS! Global Signatures saved permanently to:")
print(f"   {gcs_output_parquet}")

# Load it back to verify the exact intersection numbers
df_final = pd.read_parquet(local_output_parquet)
print("\n" + "=" * 65)
print(f"📊 FINAL SUBSET STATISTICS:")
print(f"   Original Target Molecules: {len(target_inchikeys):,}")
print(f"   Intersected Molecules:     {len(df_final):,}")
print(f"   Total Dimensions (A-E):    {df_final.shape[1] - 1:,} (Excluding InChIKey)")
print("=" * 65)
display(df_final.head())

# Clean up local workspace to save space
print("\n🧹 Cleaning up local SSD workspace...")
shutil.rmtree(local_parquet_dir, ignore_errors=True)
shutil.rmtree(duckdb_temp_dir, ignore_errors=True)
if os.path.exists(local_output_parquet): os.remove(local_output_parquet)

# Make minimol embeddings

In [ ]:
# @title
import pandas as pd
import numpy as np
import ast

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
# Inputs
minimol_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset.csv'
metadata_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

# Output (Saved in the minimol folder)
output_parquet = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset_final.parquet'

EMB_COL = 'minimol'           # The column containing the packed embeddings
JOIN_COL = 'SMILES_standard'  # The column used to link embeddings and metadata

# =====================================================================
# 2. LOAD DATA FROM GCS
# =====================================================================
print(f"📂 Loading Minimol embeddings...")
df_emb = pd.read_csv(minimol_csv)
print(f"   ✅ Loaded {len(df_emb):,} rows.")

print(f"📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)
print(f"   ✅ Loaded {len(df_meta):,} rows.")

# =====================================================================
# 3. EXPAND THE EMBEDDINGS COLUMN
# =====================================================================
if EMB_COL not in df_emb.columns:
    raise KeyError(f"❌ Could not find '{EMB_COL}'. Available columns: {list(df_emb.columns)}")

print(f"\n💥 Expanding the '{EMB_COL}' column...")

# Clean out any blank or broken embeddings to prevent crashes
df_emb = df_emb.dropna(subset=[EMB_COL])
if isinstance(df_emb[EMB_COL].iloc[0], str):
    df_emb = df_emb[df_emb[EMB_COL].str.strip() != ""]
    df_emb = df_emb[df_emb[EMB_COL].str.strip() != "[]"]
    print("   -> Parsing string representations into Python lists...")
    df_emb[EMB_COL] = df_emb[EMB_COL].apply(ast.literal_eval)

df_emb = df_emb[df_emb[EMB_COL].map(len) > 0]

# Fast extraction using numpy vectorization
print("   -> Splitting into separate float columns...")
expanded_matrix = np.vstack(df_emb[EMB_COL].values)
num_dimensions = expanded_matrix.shape[1]

dim_cols = [f"Mini_Dim_{i+1}" for i in range(num_dimensions)]
df_expanded = pd.DataFrame(expanded_matrix, columns=dim_cols, index=df_emb.index)

# Drop the old packed column and attach the expanded columns
df_emb = pd.concat([df_emb.drop(columns=[EMB_COL]), df_expanded], axis=1)
print(f"   ✅ Successfully expanded into {num_dimensions} distinct float columns!")

# =====================================================================
# 4. JOIN METADATA ONTO EMBEDDINGS
# =====================================================================
print("\n⚙️ Joining metadata onto the expanded embeddings...")

if JOIN_COL not in df_emb.columns or JOIN_COL not in df_meta.columns:
    raise KeyError(f"❌ Join column '{JOIN_COL}' must exist in BOTH files!")

# Deduplicate metadata on the join column to avoid accidental row duplication
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

# Inner join so we only keep molecules that have both embeddings and metadata
df_joined = pd.merge(df_emb, df_meta, on=JOIN_COL, how='inner')
print(f"   ✅ Join complete! Shape: {df_joined.shape[0]:,} Rows, {df_joined.shape[1]:,} Columns")

# Auto-detect the exact spelling of InChIKey (InChIKey vs Metadata_InChIKey)
INDEX_COL = next((col for col in df_joined.columns if 'inchikey' in col.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column in the joined data!")

print(f"🔑 Target index column identified as: '{INDEX_COL}'")

# =====================================================================
# 5. CLEAN UP AND SET INCHIKEY INDEX
# =====================================================================
print(f"\n🧹 Cleaning data types and setting '{INDEX_COL}' as the primary index...")

# Drop rows where InChIKey is missing
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

# Deduplicate by InChIKey (keeping the first instance)
df_joined = df_joined[~df_joined[INDEX_COL].duplicated(keep='first')]

# Set InChIKey as the primary index
df_joined.set_index(INDEX_COL, inplace=True)

# Rearrange columns to put the metadata text on the left, followed by the float dimensions
metadata_cols = df_joined.select_dtypes(exclude=['number']).columns.tolist()
embedding_cols = [c for c in df_joined.columns if c not in metadata_cols]
df_joined = df_joined[metadata_cols + embedding_cols]

# Prevent PyArrow Parquet crashes by forcing all metadata columns to be explicitly typed as strings
for col in metadata_cols:
    df_joined[col] = df_joined[col].astype(str)

# =====================================================================
# 6. SAVE TO PARQUET & VERIFY
# =====================================================================
print(f"\n💾 Saving compressed Parquet to Google Cloud Storage...")
df_joined.to_parquet(output_parquet, compression='snappy')

print(f"🎉 SUCCESS! File saved permanently to:")
print(f"   {output_parquet}")

# Display the first 3 rows to verify the index, metadata, and expanded columns
print("\n--- PREVIEW OF ASSEMBLED PARQUET ---")
display(df_joined.head(3))

# Make Bayer Embeddings

In [ ]:
import os
import duckdb
import time
import pandas as pd

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
gcs_bayer = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/source_csvs/Bayer_DINO_embeddings__Kim_et_al_subset__no_postprocessing_well_features.csv.gz'
gcs_meta = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

gcs_output = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_nonorm_final.parquet'

# Local staging paths
local_bayer = '/content/bayer_nonorm.csv.gz'
local_meta = '/content/jump_meta.csv'
local_output = '/content/BAYER_DINO_embeddings_nonorm_final.parquet'

# =====================================================================
# 2. FAST LOCAL STAGING (Bypasses DuckDB 403 Errors)
# =====================================================================
print("📥 Downloading files to local SSD for fast DuckDB processing...")
if not os.path.exists(local_bayer):
    os.system(f"gsutil cp {gcs_bayer} {local_bayer}")
if not os.path.exists(local_meta):
    os.system(f"gsutil cp {gcs_meta} {local_meta}")
print("   ✅ Files staged locally.")

# =====================================================================
# 3. AUTO-DETECT JOIN COLUMNS WITH DUCKDB
# =====================================================================
print("\n🔍 Inspecting CSV headers...")
con = duckdb.connect()

# Read just the headers of both files (takes a fraction of a second)
bayer_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_bayer}'").df()['column_name'].tolist()
meta_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_meta}'").df()['column_name'].tolist()

possible_join_cols = ['Metadata_JCP2022', 'perturbation_ID', 'perturbation_id', 'SMILES_standard', 'ID']
JOIN_COL = next((c for c in possible_join_cols if c in bayer_cols and c in meta_cols), None)

if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier column to join on! \nBayer: {bayer_cols[:5]}\nMeta: {meta_cols[:5]}")

print(f"   -> Found shared identifier: '{JOIN_COL}'")

# Find shared columns to exclude from metadata (so we don't get duplicate columns)
exclude_meta_cols = [c for c in meta_cols if c in bayer_cols and c != JOIN_COL]

# Build the dynamic EXCLUDE string for the SQL query
cols_to_exclude = ['rn', JOIN_COL] + exclude_meta_cols
exclude_str = "EXCLUDE (" + ", ".join([f'"{c}"' for c in cols_to_exclude]) + ")"

# =====================================================================
# 4. THE ZERO-RAM DUCKDB JOIN & PARQUET EXPORT (Crash-Proofed)
# =====================================================================
print("\n⚙️ Executing out-of-core LEFT JOIN directly to Parquet...")
start_time = time.time()

# 🚨 THE FIX: Added sample_size=-1, null_padding=true, and ignore_errors=true
# to the Bayer read_csv_auto function so it powers through bad lines!

query = f"""
CREATE OR REPLACE TABLE clean_meta AS
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY "{JOIN_COL}") as rn
    FROM read_csv_auto('{local_meta}', all_varchar=true)
)
WHERE rn = 1;

COPY (
    SELECT
        b.*,
        m.* {exclude_str}
    FROM read_csv_auto('{local_bayer}', sample_size=-1, null_padding=true, ignore_errors=true) AS b
    LEFT JOIN clean_meta AS m
    ON b."{JOIN_COL}" = m."{JOIN_COL}"
) TO '{local_output}' (FORMAT PARQUET, COMPRESSION 'snappy');
"""

try:
    con.execute(query)
    print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")
except Exception as e:
    raise RuntimeError(f"DuckDB Merge Error: {e}")
finally:
    con.close()

print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")

# =====================================================================
# 5. UPLOAD TO GCS & VERIFY
# =====================================================================
print(f"\n📤 Uploading fully annotated Parquet to Google Cloud Storage...")
os.system(f"gsutil cp {local_output} {gcs_output}")

# Verify the row count matches exactly!
df_final = pd.read_parquet(local_output)
print("-" * 60)
print(f"📊 Final Parquet Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print("-" * 60)

# Optional cleanup to free SSD space
os.remove(local_bayer)
os.remove(local_meta)
os.remove(local_output)

print(f"🎉 SUCCESS! File saved permanently to: {gcs_output}")

In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
# The new raw/unnormalized Bayer embeddings
bayer_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/source_csvs/Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features.csv.gz'

# The metadata CSV
metadata_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

# Output location (saving it in the main Bayer DINO folder)
output_parquet = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_norm_final.parquet'

# =====================================================================
# 2. LOAD DATA FROM GCS
# =====================================================================
print("📂 Loading Bayer DINO Embeddings (Decompressing GZIP)...")
df_bayer = pd.read_csv(bayer_csv, compression='gzip')
print(f"   ✅ Loaded {len(df_bayer):,} rows and {df_bayer.shape[1]:,} columns.")

print("📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)
print(f"   ✅ Loaded {len(df_meta):,} rows.")

# =====================================================================
# 3. AUTO-DETECT JOIN COLUMNS
# =====================================================================
print("\n🔍 Auto-detecting shared identifier columns...")

# Check for common JUMP-CP IDs to use as the bridge
possible_join_cols = ['Metadata_JCP2022', 'perturbation_ID', 'perturbation_id', 'SMILES_standard', 'ID']

JOIN_COL = next((col for col in possible_join_cols if col in df_bayer.columns and col in df_meta.columns), None)

if not JOIN_COL:
    print(f"\n❌ Error: Could not automatically find a matching ID column between the two files.")
    print(f"Bayer Columns:    {list(df_bayer.columns[:10])}")
    print(f"Metadata Columns: {list(df_meta.columns[:10])}")
    raise KeyError("Please manually check the output above to see what the shared ID column is named.")

print(f"   -> Found shared identifier for joining: '{JOIN_COL}'")

# Auto-detect the exact spelling of InChIKey (InChIKey vs Metadata_InChIKey)
INDEX_COL = next((c for c in df_meta.columns if 'inchikey' in c.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column in the Metadata CSV!")
print(f"🔑 Target index column identified as: '{INDEX_COL}'")

# =====================================================================
# 4. JOIN METADATA ONTO EMBEDDINGS
# =====================================================================
print("\n⚙️ Joining metadata onto the Bayer embeddings...")

# Deduplicate metadata on the join column to avoid accidental row duplication
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

# Drop any shared columns (except the join column) to prevent messy _x and _y duplicate columns
shared_cols = [c for c in df_meta.columns if c in df_bayer.columns and c != JOIN_COL]
if shared_cols:
    df_meta = df_meta.drop(columns=shared_cols)

# Inner join so we only keep molecules that have both embeddings and metadata
df_joined = pd.merge(df_bayer, df_meta, on=JOIN_COL, how='inner')
print(f"   ✅ Join complete! Shape: {df_joined.shape[0]:,} Rows, {df_joined.shape[1]:,} Columns")

# =====================================================================
# 5. CLEAN UP AND SET INCHIKEY INDEX
# =====================================================================
print(f"\n🧹 Cleaning data types and setting '{INDEX_COL}' as the primary index...")

# Drop rows where InChIKey is missing
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

# Deduplicate by InChIKey (keeping the first instance) to make the index mathematically unique
df_joined = df_joined[~df_joined[INDEX_COL].duplicated(keep='first')]

# Set InChIKey as the primary index
df_joined.set_index(INDEX_COL, inplace=True)

# Rearrange columns to put the metadata text on the left, followed by the float dimensions
metadata_cols = df_joined.select_dtypes(exclude=['number']).columns.tolist()
embedding_cols = [c for c in df_joined.columns if c not in metadata_cols]
df_joined = df_joined[metadata_cols + embedding_cols]

# Prevent PyArrow Parquet crashes by forcing all metadata columns (like 'plate') to be explicitly strings
for col in metadata_cols:
    df_joined[col] = df_joined[col].astype(str)

# =====================================================================
# 6. SAVE TO PARQUET & VERIFY
# =====================================================================
print(f"\n💾 Saving compressed Parquet to Google Cloud Storage...")
df_joined.to_parquet(output_parquet, compression='snappy')

print(f"🎉 SUCCESS! File saved permanently to:")
print(f"   {output_parquet}")

# Display the first 3 rows to verify the index, metadata, and expanded columns
print("\n--- PREVIEW OF ASSEMBLED PARQUET ---")
display(df_joined.head(3))

In [ ]:
import os
import duckdb
import time
import pandas as pd

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
gcs_bayer = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/source_csvs/Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features.csv.gz'
gcs_meta = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

gcs_output = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_normed_final.parquet'

# Local staging paths
local_bayer = '/content/bayer_nonorm.csv.gz'
local_meta = '/content/jump_meta.csv'
local_output = '/content/BAYER_DINO_embeddings_nonorm_final.parquet'

# =====================================================================
# 2. FAST LOCAL STAGING (Bypasses DuckDB 403 Errors)
# =====================================================================
print("📥 Downloading files to local SSD for fast DuckDB processing...")
if not os.path.exists(local_bayer):
    os.system(f"gsutil cp {gcs_bayer} {local_bayer}")
if not os.path.exists(local_meta):
    os.system(f"gsutil cp {gcs_meta} {local_meta}")
print("   ✅ Files staged locally.")

# =====================================================================
# 3. AUTO-DETECT JOIN COLUMNS WITH DUCKDB
# =====================================================================
print("\n🔍 Inspecting CSV headers...")
con = duckdb.connect()

# Read just the headers of both files (takes a fraction of a second)
bayer_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_bayer}'").df()['column_name'].tolist()
meta_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_meta}'").df()['column_name'].tolist()

possible_join_cols = ['Metadata_JCP2022', 'perturbation_ID', 'perturbation_id', 'SMILES_standard', 'ID']
JOIN_COL = next((c for c in possible_join_cols if c in bayer_cols and c in meta_cols), None)

if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier column to join on! \nBayer: {bayer_cols[:5]}\nMeta: {meta_cols[:5]}")

print(f"   -> Found shared identifier: '{JOIN_COL}'")

# Find shared columns to exclude from metadata (so we don't get duplicate columns)
exclude_meta_cols = [c for c in meta_cols if c in bayer_cols and c != JOIN_COL]

# Build the dynamic EXCLUDE string for the SQL query
cols_to_exclude = ['rn', JOIN_COL] + exclude_meta_cols
exclude_str = "EXCLUDE (" + ", ".join([f'"{c}"' for c in cols_to_exclude]) + ")"

# =====================================================================
# 4. THE ZERO-RAM DUCKDB JOIN & PARQUET EXPORT (Crash-Proofed)
# =====================================================================
print("\n⚙️ Executing out-of-core LEFT JOIN directly to Parquet...")
start_time = time.time()

# 🚨 THE FIX: Added sample_size=-1, null_padding=true, and ignore_errors=true
# to the Bayer read_csv_auto function so it powers through bad lines!

query = f"""
CREATE OR REPLACE TABLE clean_meta AS
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY "{JOIN_COL}") as rn
    FROM read_csv_auto('{local_meta}', all_varchar=true)
)
WHERE rn = 1;

COPY (
    SELECT
        b.*,
        m.* {exclude_str}
    FROM read_csv_auto('{local_bayer}', sample_size=-1, null_padding=true, ignore_errors=true) AS b
    LEFT JOIN clean_meta AS m
    ON b."{JOIN_COL}" = m."{JOIN_COL}"
) TO '{local_output}' (FORMAT PARQUET, COMPRESSION 'snappy');
"""

try:
    con.execute(query)
    print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")
except Exception as e:
    raise RuntimeError(f"DuckDB Merge Error: {e}")
finally:
    con.close()

print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")

# =====================================================================
# 5. UPLOAD TO GCS & VERIFY
# =====================================================================
print(f"\n📤 Uploading fully annotated Parquet to Google Cloud Storage...")
os.system(f"gsutil cp {local_output} {gcs_output}")

# Verify the row count matches exactly!
df_final = pd.read_parquet(local_output)
print("-" * 60)
print(f"📊 Final Parquet Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print("-" * 60)

# Optional cleanup to free SSD space
os.remove(local_bayer)
os.remove(local_meta)
os.remove(local_output)

print(f"🎉 SUCCESS! File saved permanently to: {gcs_output}")

# Filter Bayer embeddings for active only

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
PARQUET_FILE_PATH = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_normed_final.parquet'

# Output path for the filtered dataset
OUTPUT_PARQUET_PATH = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features__ACTIVE_ONLY-test2.parquet'

CONTROL_COLUMN = 'control_type'
DMSO_LABEL = 'negcon'

# =====================================================================
# 2. LOAD DATA FROM PARQUET
# =====================================================================
print(f"📂 Loading pre-sphered embeddings from GCS...")
df_bayer = pd.read_parquet(PARQUET_FILE_PATH)
print(f"   ✅ Loaded {len(df_bayer):,} rows and {df_bayer.shape[1]:,} columns.")

# =====================================================================
# 3. SEPARATE CONTROLS AND PERTURBATIONS
# =====================================================================
print("\n⚙️ Separating controls from perturbations...")

mask_dmso = df_bayer[CONTROL_COLUMN] == DMSO_LABEL
df_dmso = df_bayer[mask_dmso].copy()
df_drugs = df_bayer[~mask_dmso].copy()

# Isolate just the numerical embedding features
dmso_embs = df_dmso.select_dtypes(include=[np.number]).values
drug_embs = df_drugs.select_dtypes(include=[np.number]).values

print(f"   -> Found {len(df_dmso):,} DMSO controls.")
print(f"   -> Found {len(df_drugs):,} Drug perturbations.")

# =====================================================================
# 4. CALCULATE PHENOTYPIC ACTIVITY SCORE (PAS)
# =====================================================================
print("\n📏 Calculating distances for already-sphered data...")

# 1. Find the exact center of the DMSO sphere
dmso_centroid = np.median(dmso_embs, axis=0).reshape(1, -1)

# 2. Calculate pure Euclidean distance to the center
dmso_pas = cdist(dmso_embs, dmso_centroid, metric='euclidean').flatten()
drug_pas = cdist(drug_embs, dmso_centroid, metric='euclidean').flatten()

df_dmso['Phenotypic_Activity_Score'] = dmso_pas
df_drugs['Phenotypic_Activity_Score'] = drug_pas

# =====================================================================
# 5. SET THRESHOLD AND FILTER
# =====================================================================
THRESHOLD_PERCENTILE = 95
activity_threshold = np.percentile(dmso_pas, THRESHOLD_PERCENTILE)

print(f"\n🛑 Setting Activity Threshold at the {THRESHOLD_PERCENTILE}th percentile of DMSO.")
print(f"   -> Threshold Distance: {activity_threshold:.2f}")

# Filter the drugs
active_mask = df_drugs['Phenotypic_Activity_Score'] > activity_threshold
df_active_drugs = df_drugs[active_mask].copy()
inactive_count = len(df_drugs) - len(df_active_drugs)

print("\n🏆 FILTERING RESULTS")
print("-" * 50)
print(f"   Total Drugs:        {len(df_drugs):,}")
print(f"   ✅ Active (Kept):   {len(df_active_drugs):,} ({(len(df_active_drugs)/len(df_drugs))*100:.1f}%)")
print(f"   ❌ Inactive (Drop): {inactive_count:,} ({(inactive_count/len(df_drugs))*100:.1f}%)")
print("-" * 50)

# =====================================================================
# 6. SAVE FILTERED DATASET TO GCS
# =====================================================================
print(f"\n💾 Saving active-only filtered embeddings to Google Cloud Storage...")
# Snappy compression is excellent for Parquet files
df_active_drugs.to_parquet(OUTPUT_PARQUET_PATH, compression='snappy')
print(f"🎉 Success! Filtered dataset permanently saved to:")
print(f"   {OUTPUT_PARQUET_PATH}")

# =====================================================================
# 7. VISUALIZE THE DISTRIBUTIONS
# =====================================================================
plt.figure(figsize=(10, 6))

sns.kdeplot(dmso_pas, fill=True, color='gray', label='DMSO Controls (Noise)')
sns.kdeplot(drug_pas, fill=True, color='teal', label='Drug Perturbations')

plt.axvline(activity_threshold, color='red', linestyle='--',
            label=f'{THRESHOLD_PERCENTILE}th Percentile Threshold')

plt.title('Phenotypic Activity Score (Sphered Euclidean Distance)', fontsize=16)
plt.xlabel('Distance from DMSO Centroid', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Make OpenPhenom-384 Parquet

In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
openphenom_csv = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/single-cells_384_features/aggregated_profiles_openphenom_single_cell_384.csv.gz'
metadata_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

out_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'

DMSO_INCHIKEY = 'IAZDPXIOMUYVGZ-UHFFFAOYSA-N'
DMSO_SMILES = 'CS(C)=O'

# =====================================================================
# 2. LOAD DATA
# =====================================================================
print("📂 Loading OpenPhenom Embeddings (Decompressing GZIP)...")
df_emb = pd.read_csv(openphenom_csv, compression='gzip')

print("📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)

print("\n📊 [DIAGNOSTIC] PRE-JOIN COLUMN COUNTS:")
print(f"   -> OpenPhenom: {df_emb.shape[1]} columns")
print(f"   -> Metadata:   {df_meta.shape[1]} columns")

# =====================================================================
# 3. ORGANIZE OPENPHENOM (METADATA LEFT, EMBEDDINGS RIGHT)
# =====================================================================
# Find the exact 384 embedding columns (Assuming they are named '0', '1', '2'...)
emb_cols = [c for c in df_emb.columns if str(c).isdigit() or str(c).startswith('emb_')]
# Everything else in OpenPhenom is metadata
op_meta_cols = [c for c in df_emb.columns if c not in emb_cols]

print(f"\n⚙️ Shifting {len(op_meta_cols)} metadata columns to the front of OpenPhenom...")
df_emb = df_emb[op_meta_cols + emb_cols]

# =====================================================================
# 4. AUTO-DETECT JOIN KEYS
# =====================================================================
possible_join_keys = ['InChIKey', 'inchikey', 'Metadata_JCP2022', 'perturbation_ID', 'perturbation_id']

JOIN_COL = next((c for c in possible_join_keys if c in df_emb.columns and c in df_meta.columns), None)
if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier to join on! \nEmb Meta: {op_meta_cols}\nAnno Meta: {list(df_meta.columns[:10])}")

print(f"   -> Detected Join Key: '{JOIN_COL}'")

# Deduplicate metadata on the join key to prevent multiplying rows
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

# =====================================================================
# 5. THE LOSSLESS LEFT JOIN
# =====================================================================
print("\n🔗 Executing LEFT JOIN (Preserving all columns)...")

# We use suffixes so we can explicitly see overlapping columns (e.g. 'smiles_OP' and 'smiles_META')
df_joined = pd.merge(df_emb, df_meta, on=JOIN_COL, how='left', suffixes=('_OP', '_META'))

# Resolve overlapping columns (Prioritizing the official Metadata file, fallback to OpenPhenom)
overlap_bases = [c.replace('_META', '') for c in df_joined.columns if c.endswith('_META')]

for col in overlap_bases:
    col_op = f"{col}_OP"
    col_meta = f"{col}_META"

    # Combine them into a single clean column
    df_joined[col] = df_joined[col_meta].combine_first(df_joined[col_op])
    # Drop the messy suffix columns
    df_joined.drop(columns=[col_op, col_meta], inplace=True)

# =====================================================================
# 6. RESCUE DMSO & SET INDEX
# =====================================================================
print("\n🛡️ Rescuing DMSO controls...")

# Identify the InChIKey column for indexing
INDEX_COL = next((c for c in df_joined.columns if 'inchikey' in c.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column!")

CONTROL_COL = next((c for c in ['control_type', 'Metadata_control_type', 'pert_type'] if c in df_joined.columns), None)

if CONTROL_COL:
    is_dmso = df_joined[CONTROL_COL].astype(str).str.strip().str.lower().isin(['negcon', 'control'])
    df_joined.loc[is_dmso, INDEX_COL] = DMSO_INCHIKEY

    SMILES_COL = next((c for c in df_joined.columns if 'smiles' in c.lower()), None)
    if SMILES_COL:
        df_joined.loc[is_dmso, SMILES_COL] = DMSO_SMILES

# Clean string and explicitly drop rows that STILL don't have an InChIKey
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

print(f"⚙️ Setting '{INDEX_COL}' as the primary index...")
df_joined.set_index(INDEX_COL, inplace=True)
df_joined.index.name = 'InChIKey'

# =====================================================================
# 7. FINAL FORMATTING & SAVE
# =====================================================================
# Ensure metadata is on the left and embeddings are on the right
final_meta_cols = [c for c in df_joined.columns if c not in emb_cols]
df_joined = df_joined[final_meta_cols + emb_cols]

# Force all metadata columns to string to prevent PyArrow crashes
for col in final_meta_cols:
    df_joined[col] = df_joined[col].astype(str)

print("\n📊 [DIAGNOSTIC] POST-JOIN COLUMN COUNTS:")
print(f"   -> Final Metadata Columns:  {len(final_meta_cols)}")
print(f"   -> Final Embedding Columns: {len(emb_cols)}")
print(f"   -> TOTAL COLUMNS:           {df_joined.shape[1]}")

print(f"\n💾 Saving highly-optimized Parquet to GCS...")
df_joined.to_parquet(out_parquet, compression='snappy')

print(f"🎉 SUCCESS! File saved to:\n   {out_parquet}")
display(df_joined.head(5))

# Normalize Openphenom embeddings

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import median_abs_deviation
import time
import gcsfs

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
# The Parquet file you just assembled
input_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'

# Output normalized file
output_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized.parquet'

# The universal JUMP-CP identifier for DMSO
DMSO_JCP_ID = 'JCP2022_033924'

# Sphering regularization parameter (prevents math crashes on highly correlated features)
SPHERING_REGULARIZATION = 1e-4

# =====================================================================
# 2. LOAD DATA
# =====================================================================
print("📂 Loading OpenPhenom embeddings from GCS...")
start_time = time.time()
df = pd.read_parquet(input_parquet)
print(f"   ✅ Loaded {len(df):,} rows and {df.shape[1]:,} columns in {time.time() - start_time:.1f}s.")

# Isolate metadata vs numerical embeddings
meta_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
emb_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"   -> Found {len(meta_cols)} metadata columns and {len(emb_cols)} embedding dimensions.")

# =====================================================================
# 3. IDENTIFY DMSO CONTROLS
# =====================================================================
print(f"\n🔍 Searching metadata for DMSO identifier: '{DMSO_JCP_ID}'...")

# Create a blank mask to track DMSO rows
dmso_mask = pd.Series(False, index=df.index)

# Scan all text columns for the JCP identifier
for col in meta_cols:
    match = df[col].astype(str).str.strip().str.upper() == DMSO_JCP_ID.upper()
    if match.sum() > 0:
        print(f"   -> Found DMSO matches in column: '{col}'")
        dmso_mask = dmso_mask | match

dmso_count = dmso_mask.sum()

print("\n" + "="*50)
print(f"📊 DMSO COUNT RESULT: {dmso_count:,} wells identified.")
print("="*50)

if dmso_count == 0:
    raise ValueError(f"❌ No DMSO controls found! Could not find '{DMSO_JCP_ID}' anywhere in the metadata.")

# Separate the embedding arrays
dmso_embs = df.loc[dmso_mask, emb_cols].values
all_embs = df[emb_cols].values

# =====================================================================
# 4. STEP A: MAD-ROBUSTIZE
# =====================================================================
print("\n📐 Step A: MAD-Robustizing (Centering and Scaling)...")
start_math = time.time()

# Center the data exactly on the DMSO median
center_median = np.median(dmso_embs, axis=0)

# 🚨 MATHEMATICAL SAFEGUARD:
# If there is only 1 DMSO row (it was aggregated), its variance is 0.
# We must use the variance of the entire dataset to scale the features instead!
if dmso_count > 1:
    print("   -> Multiple DMSO rows detected. Scaling by DMSO variance (MAD).")
    spread_mad = median_abs_deviation(dmso_embs, axis=0)
else:
    print("   -> Only 1 DMSO row detected. Scaling by Global Dataset variance (MAD).")
    spread_mad = median_abs_deviation(all_embs, axis=0)

# Drop "Dead Neurons" (Features with zero variance) to prevent division by zero
valid_features = spread_mad > 1e-5
dropped_count = len(emb_cols) - np.sum(valid_features)
print(f"   -> Dropping {dropped_count} 'dead' features with ~0 variance.")

valid_emb_cols = np.array(emb_cols)[valid_features]
all_embs_clean = all_embs[:, valid_features]

center_median_clean = center_median[valid_features]
spread_mad_clean = spread_mad[valid_features]

# Apply Robust Z-Scoring to the entire dataset
all_robust = (all_embs_clean - center_median_clean) / spread_mad_clean

# If we have enough DMSO, we also need the robustized DMSO array for sphering
if dmso_count > 1:
    dmso_embs_clean = dmso_embs[:, valid_features]
    dmso_robust = (dmso_embs_clean - center_median_clean) / spread_mad_clean

# =====================================================================
# 5. STEP B: ZCA SPHERING (WHITENING)
# =====================================================================
print("🔮 Step B: Sphering (ZCA Whitening) to remove feature correlations...")

if dmso_count > 1:
    print("   -> Calculating covariance matrix from DMSO controls...")
    cov_matrix = np.cov(dmso_robust, rowvar=False)
else:
    print("   -> Single DMSO row detected. Calculating covariance matrix from Global Dataset...")
    cov_matrix = np.cov(all_robust, rowvar=False)

# Eigendecomposition of the covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Clip tiny negative eigenvalues caused by floating point precision limits
eigenvalues = np.clip(eigenvalues, a_min=0, a_max=None)

# Calculate the ZCA Sphering Matrix: W = E * D^(-1/2) * E.T
D_inv_sqrt = np.diag(1.0 / np.sqrt(eigenvalues + SPHERING_REGULARIZATION))
W_zca = eigenvectors @ D_inv_sqrt @ eigenvectors.T

# Apply the sphering transformation to the ENTIRE dataset
all_sphered = all_robust @ W_zca

print(f"   ✅ Mathematics complete in {time.time() - start_math:.2f} seconds.")

# =====================================================================
# 6. REASSEMBLE AND SAVE TO PARQUET
# =====================================================================
print("\n💾 Reassembling DataFrame and saving to Google Cloud Storage...")

# Create new dimension column names to reflect the transformation
new_dim_cols = [f"OpenPhenom_Sphered_Dim_{i+1}" for i in range(all_sphered.shape[1])]

# Build the final dataframe
df_sphered_features = pd.DataFrame(all_sphered, columns=new_dim_cols, index=df.index)

# Re-attach the metadata
df_final = pd.concat([df[meta_cols], df_sphered_features], axis=1)

# Ensure string metadata stays as strings for PyArrow compatibility
for col in meta_cols:
    df_final[col] = df_final[col].astype(str)

# Save to Parquet natively
df_final.to_parquet(output_parquet, compression='snappy')

print(f"🎉 SUCCESS! Sphered embeddings saved permanently to:\n   {output_parquet}")

# Verify output
print("\n" + "-" * 75)
print(f"📊 Final Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print(f"📌 Index Name:  {df_final.index.name}")
print("-" * 75)
display(df_final.head(3))

# Make Openphenom 1920 embeddings


In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
openphenom_csv = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/single-cells_1920_features/aggregated_profiles_openphenom_single_cell_1920.csv.gz'
metadata_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

out_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920.parquet'

DMSO_INCHIKEY = 'IAZDPXIOMUYVGZ-UHFFFAOYSA-N'
DMSO_SMILES = 'CS(C)=O'

# =====================================================================
# 2. LOAD DATA
# =====================================================================
print("📂 Loading OpenPhenom Embeddings (Decompressing GZIP)...")
df_emb = pd.read_csv(openphenom_csv, compression='gzip')

print("📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)

print("\n📊 [DIAGNOSTIC] PRE-JOIN COLUMN COUNTS:")
print(f"   -> OpenPhenom: {df_emb.shape[1]} columns")
print(f"   -> Metadata:   {df_meta.shape[1]} columns")

# =====================================================================
# 3. ORGANIZE OPENPHENOM (METADATA LEFT, EMBEDDINGS RIGHT)
# =====================================================================
# Find the exact 384 embedding columns (Assuming they are named '0', '1', '2'...)
emb_cols = [c for c in df_emb.columns if str(c).isdigit() or str(c).startswith('emb_')]
# Everything else in OpenPhenom is metadata
op_meta_cols = [c for c in df_emb.columns if c not in emb_cols]

print(f"\n⚙️ Shifting {len(op_meta_cols)} metadata columns to the front of OpenPhenom...")
df_emb = df_emb[op_meta_cols + emb_cols]

# =====================================================================
# 4. AUTO-DETECT JOIN KEYS
# =====================================================================
possible_join_keys = ['InChIKey', 'inchikey', 'Metadata_JCP2022', 'perturbation_ID', 'perturbation_id']

JOIN_COL = next((c for c in possible_join_keys if c in df_emb.columns and c in df_meta.columns), None)
if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier to join on! \nEmb Meta: {op_meta_cols}\nAnno Meta: {list(df_meta.columns[:10])}")

print(f"   -> Detected Join Key: '{JOIN_COL}'")

# Deduplicate metadata on the join key to prevent multiplying rows
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

# =====================================================================
# 5. THE LOSSLESS LEFT JOIN
# =====================================================================
print("\n🔗 Executing LEFT JOIN (Preserving all columns)...")

# We use suffixes so we can explicitly see overlapping columns (e.g. 'smiles_OP' and 'smiles_META')
df_joined = pd.merge(df_emb, df_meta, on=JOIN_COL, how='left', suffixes=('_OP', '_META'))

# Resolve overlapping columns (Prioritizing the official Metadata file, fallback to OpenPhenom)
overlap_bases = [c.replace('_META', '') for c in df_joined.columns if c.endswith('_META')]

for col in overlap_bases:
    col_op = f"{col}_OP"
    col_meta = f"{col}_META"

    # Combine them into a single clean column
    df_joined[col] = df_joined[col_meta].combine_first(df_joined[col_op])
    # Drop the messy suffix columns
    df_joined.drop(columns=[col_op, col_meta], inplace=True)

# =====================================================================
# 6. RESCUE DMSO & SET INDEX
# =====================================================================
print("\n🛡️ Rescuing DMSO controls...")

# Identify the InChIKey column for indexing
INDEX_COL = next((c for c in df_joined.columns if 'inchikey' in c.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column!")

CONTROL_COL = next((c for c in ['control_type', 'Metadata_control_type', 'pert_type'] if c in df_joined.columns), None)

if CONTROL_COL:
    is_dmso = df_joined[CONTROL_COL].astype(str).str.strip().str.lower().isin(['negcon', 'control'])
    df_joined.loc[is_dmso, INDEX_COL] = DMSO_INCHIKEY

    SMILES_COL = next((c for c in df_joined.columns if 'smiles' in c.lower()), None)
    if SMILES_COL:
        df_joined.loc[is_dmso, SMILES_COL] = DMSO_SMILES

# Clean string and explicitly drop rows that STILL don't have an InChIKey
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

print(f"⚙️ Setting '{INDEX_COL}' as the primary index...")
df_joined.set_index(INDEX_COL, inplace=True)
df_joined.index.name = 'InChIKey'

# =====================================================================
# 7. FINAL FORMATTING & SAVE
# =====================================================================
# Ensure metadata is on the left and embeddings are on the right
final_meta_cols = [c for c in df_joined.columns if c not in emb_cols]
df_joined = df_joined[final_meta_cols + emb_cols]

# Force all metadata columns to string to prevent PyArrow crashes
for col in final_meta_cols:
    df_joined[col] = df_joined[col].astype(str)

print("\n📊 [DIAGNOSTIC] POST-JOIN COLUMN COUNTS:")
print(f"   -> Final Metadata Columns:  {len(final_meta_cols)}")
print(f"   -> Final Embedding Columns: {len(emb_cols)}")
print(f"   -> TOTAL COLUMNS:           {df_joined.shape[1]}")

print(f"\n💾 Saving highly-optimized Parquet to GCS...")
df_joined.to_parquet(out_parquet, compression='snappy')

print(f"🎉 SUCCESS! File saved to:\n   {out_parquet}")
display(df_joined.head(5))

# Normalize Openphenom 1920

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import median_abs_deviation
import time
import gcsfs

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
# The Parquet file you just assembled
input_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920.parquet'

# Output normalized file
output_parquet = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized.parquet'

# The universal JUMP-CP identifier for DMSO
DMSO_JCP_ID = 'JCP2022_033924'

# Sphering regularization parameter (prevents math crashes on highly correlated features)
SPHERING_REGULARIZATION = 1e-4

# =====================================================================
# 2. LOAD DATA
# =====================================================================
print("📂 Loading OpenPhenom embeddings from GCS...")
start_time = time.time()
df = pd.read_parquet(input_parquet)
print(f"   ✅ Loaded {len(df):,} rows and {df.shape[1]:,} columns in {time.time() - start_time:.1f}s.")

# Isolate metadata vs numerical embeddings
meta_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
emb_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"   -> Found {len(meta_cols)} metadata columns and {len(emb_cols)} embedding dimensions.")

# =====================================================================
# 3. IDENTIFY DMSO CONTROLS
# =====================================================================
print(f"\n🔍 Searching metadata for DMSO identifier: '{DMSO_JCP_ID}'...")

# Create a blank mask to track DMSO rows
dmso_mask = pd.Series(False, index=df.index)

# Scan all text columns for the JCP identifier
for col in meta_cols:
    match = df[col].astype(str).str.strip().str.upper() == DMSO_JCP_ID.upper()
    if match.sum() > 0:
        print(f"   -> Found DMSO matches in column: '{col}'")
        dmso_mask = dmso_mask | match

dmso_count = dmso_mask.sum()

print("\n" + "="*50)
print(f"📊 DMSO COUNT RESULT: {dmso_count:,} wells identified.")
print("="*50)

if dmso_count == 0:
    raise ValueError(f"❌ No DMSO controls found! Could not find '{DMSO_JCP_ID}' anywhere in the metadata.")

# Separate the embedding arrays
dmso_embs = df.loc[dmso_mask, emb_cols].values
all_embs = df[emb_cols].values

# =====================================================================
# 4. STEP A: MAD-ROBUSTIZE
# =====================================================================
print("\n📐 Step A: MAD-Robustizing (Centering and Scaling)...")
start_math = time.time()

# Center the data exactly on the DMSO median
center_median = np.median(dmso_embs, axis=0)

# 🚨 MATHEMATICAL SAFEGUARD:
# If there is only 1 DMSO row (it was aggregated), its variance is 0.
# We must use the variance of the entire dataset to scale the features instead!
if dmso_count > 1:
    print("   -> Multiple DMSO rows detected. Scaling by DMSO variance (MAD).")
    spread_mad = median_abs_deviation(dmso_embs, axis=0)
else:
    print("   -> Only 1 DMSO row detected. Scaling by Global Dataset variance (MAD).")
    spread_mad = median_abs_deviation(all_embs, axis=0)

# Drop "Dead Neurons" (Features with zero variance) to prevent division by zero
valid_features = spread_mad > 1e-5
dropped_count = len(emb_cols) - np.sum(valid_features)
print(f"   -> Dropping {dropped_count} 'dead' features with ~0 variance.")

valid_emb_cols = np.array(emb_cols)[valid_features]
all_embs_clean = all_embs[:, valid_features]

center_median_clean = center_median[valid_features]
spread_mad_clean = spread_mad[valid_features]

# Apply Robust Z-Scoring to the entire dataset
all_robust = (all_embs_clean - center_median_clean) / spread_mad_clean

# If we have enough DMSO, we also need the robustized DMSO array for sphering
if dmso_count > 1:
    dmso_embs_clean = dmso_embs[:, valid_features]
    dmso_robust = (dmso_embs_clean - center_median_clean) / spread_mad_clean

# =====================================================================
# 5. STEP B: ZCA SPHERING (WHITENING)
# =====================================================================
print("🔮 Step B: Sphering (ZCA Whitening) to remove feature correlations...")

if dmso_count > 1:
    print("   -> Calculating covariance matrix from DMSO controls...")
    cov_matrix = np.cov(dmso_robust, rowvar=False)
else:
    print("   -> Single DMSO row detected. Calculating covariance matrix from Global Dataset...")
    cov_matrix = np.cov(all_robust, rowvar=False)

# Eigendecomposition of the covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Clip tiny negative eigenvalues caused by floating point precision limits
eigenvalues = np.clip(eigenvalues, a_min=0, a_max=None)

# Calculate the ZCA Sphering Matrix: W = E * D^(-1/2) * E.T
D_inv_sqrt = np.diag(1.0 / np.sqrt(eigenvalues + SPHERING_REGULARIZATION))
W_zca = eigenvectors @ D_inv_sqrt @ eigenvectors.T

# Apply the sphering transformation to the ENTIRE dataset
all_sphered = all_robust @ W_zca

print(f"   ✅ Mathematics complete in {time.time() - start_math:.2f} seconds.")

# =====================================================================
# 6. REASSEMBLE AND SAVE TO PARQUET
# =====================================================================
print("\n💾 Reassembling DataFrame and saving to Google Cloud Storage...")

# Create new dimension column names to reflect the transformation
new_dim_cols = [f"OpenPhenom_Sphered_Dim_{i+1}" for i in range(all_sphered.shape[1])]

# Build the final dataframe
df_sphered_features = pd.DataFrame(all_sphered, columns=new_dim_cols, index=df.index)

# Re-attach the metadata
df_final = pd.concat([df[meta_cols], df_sphered_features], axis=1)

# Ensure string metadata stays as strings for PyArrow compatibility
for col in meta_cols:
    df_final[col] = df_final[col].astype(str)

# Save to Parquet natively
df_final.to_parquet(output_parquet, compression='snappy')

print(f"🎉 SUCCESS! Sphered embeddings saved permanently to:\n   {output_parquet}")

# Verify output
print("\n" + "-" * 75)
print(f"📊 Final Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print(f"📌 Index Name:  {df_final.index.name}")
print("-" * 75)
display(df_final.head(3))

In [ ]:
import pandas as pd
import numpy as np

parquet_file = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'
df_test = pd.read_parquet(parquet_file)

num_cols = df_test.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = df_test.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"📊 Total Columns: {len(df_test.columns)}")
print(f"🔢 Numeric Columns Found: {len(num_cols)} -> {num_cols[:5]}")
print(f"🔤 String/Object Columns Found: {len(obj_cols)} -> {obj_cols[:10]}")

In [ ]:
import pandas as pd

# The OpenPhenom parquet file
parquet_file = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet'

print("📂 Loading metadata from OpenPhenom Parquet...")

# Read the file
df = pd.read_parquet(parquet_file)

# 1. Check using the universal JUMP-CP DMSO identifier
dmso_id = 'JCP2022_033924'
if 'Metadata_JCP2022' in df.columns:
    jcp_count = (df['Metadata_JCP2022'].astype(str).str.strip() == dmso_id).sum()
    print(f"✅ Found {jcp_count} row(s) matching '{dmso_id}' in 'Metadata_JCP2022'.")
else:
    print("❌ Column 'Metadata_JCP2022' not found.")

# 2. Check if there are any explicitly labeled "negcon" or "DMSO" rows in other metadata columns
meta_cols = df.select_dtypes(exclude=['number']).columns

print("\n🔍 Scanning all other metadata columns for 'negcon' or 'DMSO'...")
found_alt_labels = False

for col in meta_cols:
    negcon_count = df[col].astype(str).str.contains('negcon', case=False, na=False).sum()
    dmso_count = df[col].astype(str).str.contains('dmso', case=False, na=False).sum()

    if negcon_count > 0:
        print(f"   -> Found {negcon_count} row(s) with 'negcon' in '{col}'")
        found_alt_labels = True
    if dmso_count > 0:
        print(f"   -> Found {dmso_count} row(s) with 'DMSO' in '{col}'")
        found_alt_labels = True

if not found_alt_labels:
    print("   -> No explicit 'negcon' or 'DMSO' text labels found in any other columns.")

print("\n📊 Total rows in the entire dataset:", f"{len(df):,}")

# Filter out inactives from normalized data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

# =====================================================================
# 1. GCS PATHS & CONFIGURATION
# =====================================================================
# Input: The Sphered and Robustized OpenPhenom Parquet
#INPUT_PARQUET = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized.parquet'
INPUT_PARQUET = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized.parquet'

# Output: The Active-Only subset
#OUTPUT_PARQUET = 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized-ACTIVE.parquet'
OUTPUT_PARQUET =  'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized-ACTIVE.parquet'

# Universal DMSO Identifiers
DMSO_INCHIKEY = 'IAZDPXIOMUYVGZ-UHFFFAOYSA-N'
DMSO_JCP_ID = 'JCP2022_033924'

# =====================================================================
# 2. LOAD DATA
# =====================================================================
print("📂 Loading Sphered OpenPhenom embeddings from GCS...")
df = pd.read_parquet(INPUT_PARQUET)

print(f"   ✅ Loaded {len(df):,} rows and {df.shape[1]:,} columns.")

# Isolate metadata vs numerical embeddings
meta_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
emb_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# =====================================================================
# 3. IDENTIFY DMSO CONTROLS
# =====================================================================
print("\n🔍 Locating DMSO negative controls using chemical identifiers...")

# We check both the InChIKey (Index) and any JCP ID columns to ensure we catch all DMSO rows
is_dmso = pd.Series(False, index=df.index)

# 1. Check the Index (if it's InChIKey)
if df.index.name == 'InChIKey':
    is_dmso = is_dmso | (df.index == DMSO_INCHIKEY)

# 2. Check metadata columns for the JCP ID
for col in meta_cols:
    is_dmso = is_dmso | (df[col].astype(str).str.strip().str.upper() == DMSO_JCP_ID.upper())

df_dmso = df[is_dmso].copy()
df_drugs = df[~is_dmso].copy()

dmso_count = len(df_dmso)
print(f"   -> Found {dmso_count:,} DMSO control well(s).")
print(f"   -> Found {len(df_drugs):,} Drug perturbations.")

if dmso_count == 0:
    raise ValueError(f"❌ No DMSO controls found! Could not find '{DMSO_INCHIKEY}' or '{DMSO_JCP_ID}'.")

# Extract the purely numerical float arrays
dmso_embs = df_dmso[emb_cols].values
drug_embs = df_drugs[emb_cols].values

# =====================================================================
# 4. CALCULATE PHENOTYPIC ACTIVITY SCORE (PAS)
# =====================================================================
print("\n📏 Calculating Phenotypic Activity Scores (Euclidean Distance)...")

# 1. Find the exact mathematical center of the DMSO baseline
dmso_centroid = np.median(dmso_embs, axis=0).reshape(1, -1)

# 2. Calculate the pure Euclidean distance to the DMSO center
# Because the data is already sphered, Euclidean distance is mathematically perfect here!
dmso_pas = cdist(dmso_embs, dmso_centroid, metric='euclidean').flatten()
drug_pas = cdist(drug_embs, dmso_centroid, metric='euclidean').flatten()

df_dmso['Phenotypic_Activity_Score'] = dmso_pas
df_drugs['Phenotypic_Activity_Score'] = drug_pas

# =====================================================================
# 5. SET DYNAMIC THRESHOLD AND FILTER
# =====================================================================
# If we have a healthy population of DMSO replicates, use the 95th percentile of DMSO noise
if dmso_count >= 10:
    THRESHOLD_PERCENTILE = 95
    activity_threshold = np.percentile(dmso_pas, THRESHOLD_PERCENTILE)
    print(f"\n🛑 Setting Activity Threshold at the {THRESHOLD_PERCENTILE}th percentile of DMSO noise.")

# If the dataset was pre-aggregated and we only have 1 or a few DMSO rows, we drop the bottom 5% of drugs
else:
    DROP_BOTTOM_PERCENT = 5
    activity_threshold = np.percentile(drug_pas, DROP_BOTTOM_PERCENT)
    print(f"\n🛑 Only {dmso_count} DMSO row(s) detected. Setting threshold to drop the bottom {DROP_BOTTOM_PERCENT}% of inactive drugs.")

print(f"   -> Threshold Distance: {activity_threshold:.2f}")

# Filter the drugs to keep ONLY the active ones
active_mask = df_drugs['Phenotypic_Activity_Score'] > activity_threshold

# Apply to the original dataframe to preserve all pristine formatting/columns
df_active_drugs = df_drugs[active_mask].copy()

inactive_count = len(df_drugs) - len(df_active_drugs)

print("\n🏆 FILTERING RESULTS")
print("-" * 50)
print(f"   Total Drugs:        {len(df_drugs):,}")
print(f"   ✅ Active (Kept):   {len(df_active_drugs):,} ({(len(df_active_drugs)/len(df_drugs))*100:.1f}%)")
print(f"   ❌ Inactive (Drop): {inactive_count:,} ({(inactive_count/len(df_drugs))*100:.1f}%)")
print("-" * 50)

# =====================================================================
# 6. SAVE FILTERED DATASET TO GCS
# =====================================================================
print(f"\n💾 Saving active-only filtered embeddings to Google Cloud Storage...")

# Drop the PAS score before saving so it doesn't get accidentally treated as an embedding dimension
df_active_drugs = df_active_drugs.drop(columns=['Phenotypic_Activity_Score'], errors='ignore')

# Save securely to GCS
df_active_drugs.to_parquet(OUTPUT_PARQUET, compression='snappy')
print(f"🎉 Success! Filtered dataset permanently saved to:\n   {OUTPUT_PARQUET}")

# =====================================================================
# 7. VISUALIZE THE DISTRIBUTIONS
# =====================================================================
plt.figure(figsize=(10, 6))

# If there is enough DMSO to form a distribution, plot it
if dmso_count > 1:
    sns.kdeplot(dmso_pas, fill=True, color='gray', label='DMSO Controls (Noise)', cut=0)

sns.kdeplot(drug_pas, fill=True, color='teal', label='Drug Perturbations', cut=0)

plt.axvline(activity_threshold, color='red', linestyle='--', linewidth=2,
            label=f'Activity Threshold ({activity_threshold:.2f})')

plt.title('OpenPhenom Phenotypic Activity Score Distribution', fontsize=16, pad=15)
plt.xlabel('Euclidean Distance from DMSO Baseline', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

#Cluster Chemical representations

In [ ]:
# @title
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com gcsfs -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.preprocessing import normalize

# Import everything from RAPIDS (GPU Acceleration)
from cuml.manifold import UMAP as cuUMAP
from cuml.cluster import HDBSCAN as cuHDBSCAN
from cuml.decomposition import PCA as cuPCA

# =====================================================================
# 1. LOAD DATA & ISOLATE MINIMOL EMBEDDINGS
# =====================================================================
print("📥 Loading Minimol Parquet from Google Cloud Storage...")

gcs_input_path = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset_final.parquet'
gcs_output_csv = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_clusters.csv'

df = pd.read_parquet(gcs_input_path)

print("⚙️ Isolating numerical embeddings for math operations...")

# List all possible metadata/text columns we want to ignore
metadata_cols =[
    'SMILES', 'SMILES_standard', 'smiles', 'CID', 'PubChem_CID',
    'InChIKey', 'Metadata_InChIKey', 'Metadata_JCP2022', 'perturbation_ID',
    'plate', 'well', 'Batch', 'Metadata_broad_sample'
]

# Keep only the actual embedding columns
emb_cols =[c for c in df.columns if c not in metadata_cols]

# Safely extract ONLY the floating-point columns
df_numeric = df[emb_cols].select_dtypes(include=[np.number]).copy()
df_numeric = df_numeric.dropna()

# 🚨 Explicitly save the column names so we can drop them from the final metadata table!
dim_cols = df_numeric.columns.tolist()

X = df_numeric.values
num_molecules = len(X)
num_dimensions = X.shape[1]

print(f"   ✅ Using ALL {num_molecules:,} valid molecules and {num_dimensions:,} dimensions.")

if num_molecules == 0:
    raise ValueError("DataFrame is completely empty! Check the data loading step.")

# Ensure we have an InChIKey column for the final export
if 'InChIKey' not in df.columns and df.index.name != 'InChIKey':
    # Attempt auto-detection
    inchikey_col = next((c for c in df.columns if 'inchikey' in c.lower()), None)
    if inchikey_col:
        df = df.rename(columns={inchikey_col: 'InChIKey'})
    else:
        print("⚠️ Warning: No 'InChIKey' column detected in the original dataframe. It will be missing from the output CSV.")

# =====================================================================
# 2. GPU-ACCELERATED UMAP VISUALIZATION
# =====================================================================
print("\n🌌 Running GPU UMAP on ALL molecules...")
start_umap = time.time()

# cuUMAP runs natively on the GPU. output_type='numpy' allows Matplotlib to plot it!
reducer = cuUMAP(n_components=2, metric='cosine', n_neighbors=15, min_dist=0.1, random_state=42, output_type='numpy')
embedding_2d = reducer.fit_transform(X)

print(f"   -> GPU UMAP completed in {time.time() - start_umap:.2f} seconds.")

plt.figure(figsize=(10, 8))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], alpha=0.5, s=10, color='#2C7BB6') # Blue for minimol
plt.title(f'1. UMAP Projection of Minimol chemical space (N={num_molecules:,})', fontsize=16, pad=15)
plt.xlabel('UMAP 1', fontsize=12)
plt.ylabel('UMAP 2', fontsize=12)
plt.grid(False)
sns.despine()
plt.tight_layout()
plt.show()

# =====================================================================
# 3. GPU-ACCELERATED HDBSCAN CLUSTERING
# =====================================================================
print("\n🚀 Running GPU HDBSCAN on FULL High-Dimensional Space...")
start_hdbscan = time.time()

# L2 Normalize vectors so Euclidean distance acts exactly like Cosine distance
X_norm = normalize(X, norm='l2')

# Tuned for tight clusters
clusterer = cuHDBSCAN(
    min_cluster_size=15,
    min_samples=30,
    metric='euclidean',
    cluster_selection_method='leaf',
    output_type='numpy'
)
cluster_labels = clusterer.fit_predict(X_norm)

num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
num_noise = np.sum(cluster_labels == -1)

print(f"   -> GPU HDBSCAN completed in {time.time() - start_hdbscan:.2f} seconds.")
print(f"   -> Found {num_clusters} distinct clusters.")
print(f"   -> {num_noise:,} molecules ({(num_noise/num_molecules)*100:.1f}%) classified as background noise.")

# Plot UMAP colored by GPU HDBSCAN Clusters
plt.figure(figsize=(10, 8))
noise_mask = (cluster_labels == -1)
plt.scatter(embedding_2d[noise_mask, 0], embedding_2d[noise_mask, 1],
            color='lightgray', alpha=0.3, s=10, label='Noise (-1)')

cluster_mask = (cluster_labels != -1)
plt.scatter(embedding_2d[cluster_mask, 0], embedding_2d[cluster_mask, 1],
            c=cluster_labels[cluster_mask], cmap='Spectral', alpha=0.7, s=15)

plt.title(f'2. GPU HDBSCAN Clusters (n={num_clusters}) mapped on chemical space UMAP', fontsize=16, pad=15)
plt.xlabel('UMAP 1', fontsize=12)
plt.ylabel('UMAP 2', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

# =====================================================================
# 4. GPU PCA + HIERARCHICAL CLUSTERING
# =====================================================================
print("\n🌳 Running GPU PCA and Hierarchical Clustering on valid HDBSCAN groups...")

X_clustered_only = X[cluster_mask]
labels_clustered_only = cluster_labels[cluster_mask]
num_valid_mols = len(X_clustered_only)

print(f"   -> Stripped noise. Processing {num_valid_mols:,} clustered molecules.")

if num_valid_mols >= 2:
    # PCA without random_state for cuML compatibility
    print("   -> Reducing dimensionality to 20 Principal Components on GPU...")
    pca = cuPCA(n_components=20, output_type='numpy')
    X_pca = pca.fit_transform(X_clustered_only)

    variance_kept = np.sum(pca.explained_variance_ratio_) * 100
    print(f"   -> PCA complete. Preserved {variance_kept:.1f}% of variance.")

    df_hc_pca = pd.DataFrame(X_pca, columns=[f'PC {i+1}' for i in range(20)])

    # Map the HDBSCAN cluster IDs to their respective colors for the rows
    unique_labels = np.unique(labels_clustered_only)
    palette = sns.color_palette("Spectral", len(unique_labels))
    lut = dict(zip(unique_labels, palette))
    row_colors = pd.Series(labels_clustered_only).map(lut).values

    # Calculate expected RAM usage for SciPy
    expected_ram_gb = (num_valid_mols * (num_valid_mols - 1) / 2 * 8) / (1024**3)
    print(f"   -> Expected RAM allocation for SciPy distance matrix: ~{expected_ram_gb:.2f} GB")
    print("   -> Calculating linkage and drawing Clustermap...")

    g = sns.clustermap(
        df_hc_pca, metric="cosine", method="average", cmap="coolwarm", figsize=(12, 10),
        standard_scale=1, row_cluster=True, col_cluster=False,
        row_colors=row_colors, yticklabels=False, xticklabels=True
    )

    g.fig.suptitle(f'3. Hierarchical Clustering (ALL {num_valid_mols:,} Clustered Mols, 20-D PCA)', y=1.02, fontsize=16)
    g.ax_heatmap.set_xlabel(f'Principal Components (Preserved {variance_kept:.1f}% Variance)')
    g.ax_heatmap.set_ylabel('Molecules (Color-coded by HDBSCAN Cluster)')
    plt.show()

# =====================================================================
# 5. ASSEMBLE THE FINAL METADATA & CLUSTER TABLE
# =====================================================================
print("\n📋 Assembling the final Metadata + Cluster Membership table...")

# We use df.loc to ensure we only grab rows that survived the dropna() step
df_results = df.loc[df_numeric.index].copy()

# Insert the cluster labels and UMAP coordinates
df_results.insert(0, 'HDBSCAN_Cluster_ID', cluster_labels)
df_results.insert(1, 'UMAP_X', embedding_2d[:, 0])
df_results.insert(2, 'UMAP_Y', embedding_2d[:, 1])

# Make sure the index is explicitly preserved as a column
if df_results.index.name:
    df_results = df_results.reset_index()

# 🚨 THE FIX: Ensure InChIKey is explicitly included in the final output
# If InChIKey isn't present in the resulting dataframe, this will alert the user.
if 'InChIKey' not in df_results.columns:
     print("\n⚠️ WARNING: 'InChIKey' column was not found during export! The CSV will lack InChIKeys.")
else:
     print(f"   -> InChIKey column successfully verified for export.")

# Drop all the raw dimensions so the CSV output is extremely clean
df_metadata_only = df_results.drop(columns=dim_cols, errors='ignore')

# Sort so clustered molecules are at the top, grouped by Cluster_ID, and noise is at the bottom
df_metadata_only['is_noise'] = (df_metadata_only['HDBSCAN_Cluster_ID'] == -1)
df_metadata_only = df_metadata_only.sort_values(by=['is_noise', 'HDBSCAN_Cluster_ID']).drop(columns=['is_noise'])

print(f"💾 Saving the final table to Google Cloud Storage...")
df_metadata_only.to_csv(gcs_output_csv, index=False)

print(f"🎉 SUCCESS! File saved to:")
print(f"   {gcs_output_csv}")

print("\n" + "="*80)
print("🏆 FINAL CLUSTER MEMBERSHIP TABLE (Preview)")
print("="*80)
display(df_metadata_only.head(15))

In [ ]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com gcsfs -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.preprocessing import normalize

# Import everything from RAPIDS (GPU Acceleration)
from cuml.manifold import UMAP as cuUMAP
from cuml.cluster import HDBSCAN as cuHDBSCAN
from cuml.decomposition import PCA as cuPCA

# =====================================================================
# 1. LOAD DATA & ISOLATE MINIMOL EMBEDDINGS
# =====================================================================
print("📥 Loading Minimol Parquet from Google Cloud Storage...")

gcs_input_path = 'gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/parquets/JUMPCP_Sign3_CC_final.parquet'
gcs_output_csv = 'gs://calico-reference-embeddings/Chemical_Checker_Embeddings/parquets/sign3_clusters.csv'

df = pd.read_parquet(gcs_input_path)

print("⚙️ Isolating numerical embeddings for math operations...")

# List all possible metadata/text columns we want to ignore
metadata_cols =[
    'SMILES', 'SMILES_standard', 'smiles', 'CID', 'PubChem_CID',
    'InChIKey', 'Metadata_InChIKey', 'Metadata_JCP2022', 'perturbation_ID',
    'plate', 'well', 'Batch', 'Metadata_broad_sample'
]

# Keep only the actual embedding columns
emb_cols =[c for c in df.columns if c not in metadata_cols]

# Safely extract ONLY the floating-point columns
df_numeric = df[emb_cols].select_dtypes(include=[np.number]).copy()
df_numeric = df_numeric.dropna()

# 🚨 Explicitly save the column names so we can drop them from the final metadata table!
dim_cols = df_numeric.columns.tolist()

X = df_numeric.values
num_molecules = len(X)
num_dimensions = X.shape[1]

print(f"   ✅ Using ALL {num_molecules:,} valid molecules and {num_dimensions:,} dimensions.")

if num_molecules == 0:
    raise ValueError("DataFrame is completely empty! Check the data loading step.")

# Ensure we have an InChIKey column for the final export
if 'InChIKey' not in df.columns and df.index.name != 'InChIKey':
    # Attempt auto-detection
    inchikey_col = next((c for c in df.columns if 'inchikey' in c.lower()), None)
    if inchikey_col:
        df = df.rename(columns={inchikey_col: 'InChIKey'})
    else:
        print("⚠️ Warning: No 'InChIKey' column detected in the original dataframe. It will be missing from the output CSV.")

# =====================================================================
# 2. GPU-ACCELERATED UMAP VISUALIZATION
# =====================================================================
print("\n🌌 Running GPU UMAP on ALL molecules...")
start_umap = time.time()

# cuUMAP runs natively on the GPU. output_type='numpy' allows Matplotlib to plot it!
reducer = cuUMAP(n_components=2, metric='cosine', n_neighbors=15, min_dist=0.1, random_state=42, output_type='numpy')
embedding_2d = reducer.fit_transform(X)

print(f"   -> GPU UMAP completed in {time.time() - start_umap:.2f} seconds.")

plt.figure(figsize=(10, 8))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], alpha=0.5, s=10, color='#2C7BB6') # Blue for minimol
plt.title(f'1. UMAP Projection of ChemChecker (Sign3) chemical space (N={num_molecules:,})', fontsize=16, pad=15)
plt.xlabel('UMAP 1', fontsize=12)
plt.ylabel('UMAP 2', fontsize=12)
plt.grid(False)
sns.despine()
plt.tight_layout()
plt.show()

# =====================================================================
# 3. GPU-ACCELERATED HDBSCAN CLUSTERING
# =====================================================================
print("\n🚀 Running GPU HDBSCAN on FULL High-Dimensional Space...")
start_hdbscan = time.time()

# L2 Normalize vectors so Euclidean distance acts exactly like Cosine distance
X_norm = normalize(X, norm='l2')

# Tuned for tight clusters
clusterer = cuHDBSCAN(
    min_cluster_size=15,
    min_samples=30,
    metric='euclidean',
    cluster_selection_method='leaf',
    output_type='numpy'
)
cluster_labels = clusterer.fit_predict(X_norm)

num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
num_noise = np.sum(cluster_labels == -1)

print(f"   -> GPU HDBSCAN completed in {time.time() - start_hdbscan:.2f} seconds.")
print(f"   -> Found {num_clusters} distinct clusters.")
print(f"   -> {num_noise:,} molecules ({(num_noise/num_molecules)*100:.1f}%) classified as background noise.")

# Plot UMAP colored by GPU HDBSCAN Clusters
plt.figure(figsize=(10, 8))
noise_mask = (cluster_labels == -1)
plt.scatter(embedding_2d[noise_mask, 0], embedding_2d[noise_mask, 1],
            color='lightgray', alpha=0.3, s=10, label='Noise (-1)')

cluster_mask = (cluster_labels != -1)
plt.scatter(embedding_2d[cluster_mask, 0], embedding_2d[cluster_mask, 1],
            c=cluster_labels[cluster_mask], cmap='Spectral', alpha=0.7, s=15)

plt.title(f'2. GPU HDBSCAN Clusters (n={num_clusters}) mapped on chemical space UMAP', fontsize=16, pad=15)
plt.xlabel('UMAP 1', fontsize=12)
plt.ylabel('UMAP 2', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

# =====================================================================
# 4. GPU PCA + HIERARCHICAL CLUSTERING
# =====================================================================
print("\n🌳 Running GPU PCA and Hierarchical Clustering on valid HDBSCAN groups...")

X_clustered_only = X[cluster_mask]
labels_clustered_only = cluster_labels[cluster_mask]
num_valid_mols = len(X_clustered_only)

print(f"   -> Stripped noise. Processing {num_valid_mols:,} clustered molecules.")

if num_valid_mols >= 2:
    # PCA without random_state for cuML compatibility
    print("   -> Reducing dimensionality to 20 Principal Components on GPU...")
    pca = cuPCA(n_components=20, output_type='numpy')
    X_pca = pca.fit_transform(X_clustered_only)

    variance_kept = np.sum(pca.explained_variance_ratio_) * 100
    print(f"   -> PCA complete. Preserved {variance_kept:.1f}% of variance.")

    df_hc_pca = pd.DataFrame(X_pca, columns=[f'PC {i+1}' for i in range(20)])

    # Map the HDBSCAN cluster IDs to their respective colors for the rows
    unique_labels = np.unique(labels_clustered_only)
    palette = sns.color_palette("Spectral", len(unique_labels))
    lut = dict(zip(unique_labels, palette))
    row_colors = pd.Series(labels_clustered_only).map(lut).values

    # Calculate expected RAM usage for SciPy
    expected_ram_gb = (num_valid_mols * (num_valid_mols - 1) / 2 * 8) / (1024**3)
    print(f"   -> Expected RAM allocation for SciPy distance matrix: ~{expected_ram_gb:.2f} GB")
    print("   -> Calculating linkage and drawing Clustermap...")

    g = sns.clustermap(
        df_hc_pca, metric="cosine", method="average", cmap="coolwarm", figsize=(12, 10),
        standard_scale=1, row_cluster=True, col_cluster=False,
        row_colors=row_colors, yticklabels=False, xticklabels=True
    )

    g.fig.suptitle(f'3. Hierarchical Clustering (ALL {num_valid_mols:,} Clustered Mols, 20-D PCA)', y=1.02, fontsize=16)
    g.ax_heatmap.set_xlabel(f'Principal Components (Preserved {variance_kept:.1f}% Variance)')
    g.ax_heatmap.set_ylabel('Molecules (Color-coded by HDBSCAN Cluster)')
    plt.show()

# =====================================================================
# 5. ASSEMBLE THE FINAL METADATA & CLUSTER TABLE
# =====================================================================
print("\n📋 Assembling the final Metadata + Cluster Membership table...")

# We use df.loc to ensure we only grab rows that survived the dropna() step
df_results = df.loc[df_numeric.index].copy()

# Insert the cluster labels and UMAP coordinates
df_results.insert(0, 'HDBSCAN_Cluster_ID', cluster_labels)
df_results.insert(1, 'UMAP_X', embedding_2d[:, 0])
df_results.insert(2, 'UMAP_Y', embedding_2d[:, 1])

# Make sure the index is explicitly preserved as a column
if df_results.index.name:
    df_results = df_results.reset_index()

# 🚨 THE FIX: Ensure InChIKey is explicitly included in the final output
# If InChIKey isn't present in the resulting dataframe, this will alert the user.
if 'InChIKey' not in df_results.columns:
     print("\n⚠️ WARNING: 'InChIKey' column was not found during export! The CSV will lack InChIKeys.")
else:
     print(f"   -> InChIKey column successfully verified for export.")

# Drop all the raw dimensions so the CSV output is extremely clean
df_metadata_only = df_results.drop(columns=dim_cols, errors='ignore')

# Sort so clustered molecules are at the top, grouped by Cluster_ID, and noise is at the bottom
df_metadata_only['is_noise'] = (df_metadata_only['HDBSCAN_Cluster_ID'] == -1)
df_metadata_only = df_metadata_only.sort_values(by=['is_noise', 'HDBSCAN_Cluster_ID']).drop(columns=['is_noise'])

print(f"💾 Saving the final table to Google Cloud Storage...")
df_metadata_only.to_csv(gcs_output_csv, index=False)

print(f"🎉 SUCCESS! File saved to:")
print(f"   {gcs_output_csv}")

print("\n" + "="*80)
print("🏆 FINAL CLUSTER MEMBERSHIP TABLE (Preview)")
print("="*80)
display(df_metadata_only.head(15))

# File paths of parquet files


In [ ]:
#minimol: gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset_final.parquet
#minimol cluster: gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_clusters.csv


#chemical checker sign2: gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/parquets/JUMPCP_CC_Sign2_Global_Signatures.parquet
#chemical checker sign3: gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/parquets/JUMPCP_Sign3_CC_final.parquet
#chemical checker sign3 clustered: gs://calico-reference-embeddings/Chemical_Checker_Embeddings/parquets/sign3_clusters.csv

#bayer DINO: gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_nonorm_final.parquet
#bayer DINO+post processing: gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_normed_final.parquet
#bayer DINO+post processing ACTIVES: gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/Bayer_DINO_embeddings_normed_final-actives_only.parquet

#openphenom-384: gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384.parquet
#openphenom-384 sphered+madrobustize: gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized.parquet
#openphenom-1920: gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920.parquet
#openphenom-1920 sphered+madrobustize: gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized.parquet
#openphenom-384 sphered+madrobustize ACTIVES: 'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_384_sphered_mad_robustized-ACTIVE.parquet'
#openphenom-1920 sphered+madrobustize ACTIVES:'gs://calico-reference-embeddings/embeddings/JUMP_Bayer_compounds/OpenPhenom/parquets/openphenom_1920_sphered_mad_robustized-ACTIVE.parquet'

